## 2024-11-08

### Authors
* Nicole Tin (nicole@velexi.com)


### Future Ideas
* benchmark with professional dermatologists estimate (+-10 yrs)
  * professional skin attribute assessment
     * is this feature something we can model?
* inter-group variability
* ML nor doctors may not be good at identifying biological age
* scientifically, there may be more group variability
* one person in training/test split as L/R hands
  * should it be strictly L or R? Is this data leakage?
  * train only on L, test on R, would outcomes be perfect? high error? 


### EDA ideas (age vs sun exposure)
* L vs. R
* inter age

### Current Tasks:
Email Dr. Powers with updates, and updated correlation findings
  - Craft interesting questions ("a research proposal"; incorporate literature)
     - Progress so far
     - Interesting finds
     - Possible avenues in coming months
     - Literature , annotator agreement
  - Some work involved in formulating an interesting research question
- Propose interesting questions (start a discussion) with derms to generate interest
- Avenues for collaboration and investigation on each side

### Preparations

In [33]:
# Paths
src_dir = '/Users/nicole/Documents/DermaML_local/hawkeye-hands-2024-07-29'
image_dir = '/processed_images/'
csv_file = '/metadata.csv'
texture_features_file = '/Users/nicole/Documents/GitHub/DermaML/notebooks/2024-08-01_NT_Hawkeye-Hands-Texture-Features.csv'

In [1]:
# --- Imports

# Internal library
from dermaml.data import read_local
from dermaml.CNN.softmax import Softmax
from dermaml.CNN.maxpool import MaxPool2
from dermaml.CNN.conv import Conv3x3

# Standard library
import random 
import math
from datetime import datetime
# External packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Entropy
random.seed(42)
np.random.seed(42)

np.set_printoptions(precision=3, suppress=True)

In [4]:
metadata = pd.read_csv(src_dir+ csv_file)
metadata.loc[:, 'Age'] = 2024-metadata['birth_year']
valid_image_fnames_df = pd.DataFrame(metadata.set_index('Age').loc[:, ['right_hand_image_file', 'left_hand_image_file']].stack()).reset_index()
valid_image_fnames_df.columns = ['Age', 'handedness', 'filename']
valid_image_fnames_df.loc[:,'filename'] = valid_image_fnames_df['filename'].apply(lambda x: x[:-5])
valid_image_fnames_df = valid_image_fnames_df.drop(index=[12,13])
valid_image_fnames = valid_image_fnames_df['filename'].to_numpy()
y = valid_image_fnames_df.loc[:, 'Age'].to_numpy()

In [40]:
texture_features = pd.read_csv(texture_features_file)
texture_features = texture_features.drop(columns=['Unnamed: 0',])

# data cleaning and preprocessing
outliers = texture_features[texture_features.Age > 200]
texture_features = texture_features.drop(index=outliers.index)
texture_features.loc[:, 'Age'] = np.around(texture_features.loc[:, 'Age']/5, decimals=0)*5

# take the norm of the top features
top_shap = ["relative_redness_std", "GLCM_SumEntropy_Mean_wrinkles_pyfeats", "skin_folds_hessian", "correlation_scikit", "lbp_6", "lbp_5", "lbp_4", "lbp_3",]
norm_texture = (texture_features[top_shap] - texture_features[top_shap].mean())/texture_features[top_shap].std()
norm_texture.loc[:, "Age"] = texture_features.loc[:, 'Age']

### Variability in Texture wrt Age

In [50]:
norm_texture.value_counts('Age')

Age
20     110
25      80
30      54
35      42
40      36
65      34
70      34
60      32
45      28
50      26
55      22
75      18
80      18
90      16
95      10
85      10
0        2
100      2
Name: count, dtype: int64

In [51]:
norm_texture.groupby('Age').mean()

KeyError: 'Age'

In [31]:
norm_texture.groupby('Age').std()

,relative_redness_std,GLCM_SumEntropy_Mean_wrinkles_pyfeats,skin_folds_hessian,correlation_scikit,lbp_6,lbp_5,lbp_4,lbp_3
Age,,,,,,,,
0,0.055363,0.187465,0.193324,0.001218,0.022084,0.015620,0.033865,0.021567
20,1.100723,0.988529,1.180667,0.332556,0.594469,0.426414,0.432416,0.622894
25,0.800860,0.749663,0.871230,0.984852,0.736324,0.617238,0.585893,0.691708
30,0.882679,1.018700,0.816312,0.987432,0.879393,0.769414,0.771109,0.863378
35,1.542750,1.322035,1.717557,0.993639,0.715157,0.680411,0.652978,0.751077
40,1.063774,0.818493,0.565117,0.773771,0.722018,0.608792,0.589339,0.701182
45,1.230381,1.135020,1.431418,0.967347,1.026942,0.917627,0.883019,0.934040
50,0.871240,0.899038,0.628973,0.862001,0.841812,0.795876,0.798574,0.807845
55,0.495330,0.894778,0.350586,1.215101,0.882005,0.728157,0.605790,0.805946


### w Metadata

need to merge filenames and L/R filenames

In [47]:
metadata.columns

Index(['record_id', 'gender', 'gender_specify', 'birth_year',
       'sex_assigned_at_birth', 'sex_assigned_at_birth_specify',
       'race_ethnicity', 'race_ethnicity_specify', 'ethnicity', 'handedness',
       'occupation', 'driving_time', 'sun_exposure', 'sunscreen_use', 'region',
       'region_specify', 'left_hand_image_file', 'right_hand_image_file',
       'form_complete', 'Age'],
      dtype='object')

In [48]:
metadata.melt(value_vars=['left_hand_image_file', 'right_hand_image_file'], var_name='hand_file', value_name='filename')

,hand_file,filename
0,left_hand_image_file,85e5965e-adff-4632-a9b9-3dbad1af9f39.jpeg
1,left_hand_image_file,16b45eff-7f1d-4437-b8fa-c7d4adbf4f92.jpeg
2,left_hand_image_file,dec83632-3864-47f9-9c03-06ea612bbd27.jpeg
3,left_hand_image_file,051963f3-269e-447c-b253-8c9c33ebf790.jpeg
4,left_hand_image_file,6c550aae-8181-41aa-829c-b42d255a9e2f.jpeg
...,...,...
621,right_hand_image_file,0ed10075-1b8b-4f30-957b-9871b8960320.jpeg
622,right_hand_image_file,b1b01048-fe7c-4df5-a17f-a9c1e507d344.jpeg
623,right_hand_image_file,0f4fc4e0-912f-49fd-ac66-ef040b14903f.jpeg
624,right_hand_image_file,37c2116c-9c15-4160-aa6c-e391923b037f.jpeg


In [44]:
valid_image_fnames_df.join(texture_features[top_shap + ['Age', 'filename']].set_index('filename'), on='filename', how='inner')

ValueError: columns overlap but no suffix specified: Index(['Age'], dtype='object')